# Lecture: Variational Autoencoder (VAE)

The standard autoencoder (notebook 63) maps each input to a **single point** in latent space. This works well for reconstruction and compression, but makes the latent space unreliable for *generation*: there is no guarantee that an arbitrary point $\mathbf{z}$ sampled from some distribution actually decodes to a realistic image.

A **Variational Autoencoder** (Kingma & Welling, 2013) fixes this by making the encoder *probabilistic*. Instead of a point, the encoder produces the parameters of a Gaussian posterior:

$$q_\psi(\mathbf{z} \mid \mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}_\psi(\mathbf{x}),\, \text{diag}(\boldsymbol{\sigma}^2_\psi(\mathbf{x})))$$

Training maximises the **Evidence Lower BOund (ELBO)**:

$$ELBO(\psi, \theta) = \underbrace{\mathbb{E}_{q_\psi(\mathbf{z}|\mathbf{x})}[\log p_\theta(\mathbf{x} \mid \mathbf{z}])}_{\text{reconstruction}} - \underbrace{D_{KL}\bigl(q_\psi(\mathbf{z}|\mathbf{x}),\, p_\theta(\mathbf{z})\bigr)}_{\text{regularisation}}$$

- **Reconstruction term**: the decoder should reproduce the input well — identical to the AE objective.
- **KL term**: the posterior $q_\psi(\mathbf{z}|\mathbf{x})$ is pulled towards the standard normal prior $p_\theta(\mathbf{z}) = \mathcal{N}(\mathbf{0}, \mathbf{I})$. This regularises the latent space and enables sampling at test time.

Because sampling from $q_\psi$ is non-differentiable, we use the **reparametrisation trick**:

$$\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

This moves the randomness into a fixed noise variable $\boldsymbol{\varepsilon}$, so gradients flow through $\boldsymbol{\mu}$ and $\boldsymbol{\sigma}$ as usual.

For the Gaussian case, the KL term has a closed-form solution per latent dimension $j$:

$$D_{KL} = -\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

Run the following cell only if you are working with Google Colab to copy the required .py file into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/06-Generative_Image_Models/VAE.py ./

### Data Preparation

We use the full MNIST training set (60,000 images), identical to the autoencoder notebook. Pixel values are normalised to $[0, 1]$.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

### Model Architecture

The VAE shares the same convolutional backbone as the autoencoder from notebook 63. The key differences are in the encoder output:

- **Autoencoder encoder**: one linear head → $\mathbf{z}$
- **VAE encoder**: two linear heads → $\boldsymbol{\mu}$ and $\log \boldsymbol{\sigma}^2$

The decoder is identical. The `forward()` method additionally returns $\boldsymbol{\mu}$ and $\log \boldsymbol{\sigma}^2$ so the loss function can compute the KL term.

In [ ]:
from VAE import VAE

LATENT_DIM = 2

_vae = VAE(latent_dim=LATENT_DIM)
_x   = torch.zeros(4, 1, 28, 28)
_x_hat, _mu, _log_var = _vae(_x)

print("Reconstruction shape:", _x_hat.shape)    # (4, 1, 28, 28)
print("mu shape:            ", _mu.shape)        # (4, 2)
print("log_var shape:       ", _log_var.shape)   # (4, 2)

### ELBO Loss

The total loss is the **negative ELBO**, i.e. we *minimise*:

$$-ELBO = \underbrace{-\mathbb{E}_{q_\psi(z|x)}[\log p_\theta(x|z)]}_{\text{reconstruction loss}} + \underbrace{D_{KL}\bigl(q_\psi(\mathbf{z}|\mathbf{x}),\, p_\theta(\mathbf{z})\bigr)}_{\text{KL regularisation}}$$

For a Gaussian decoder $p_\theta(x|z)$ the reconstruction term reduces to the MSE. The closed-form KL for a diagonal Gaussian posterior $q_\psi = \mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))$ against $p_\theta(z) = \mathcal{N}(0, I)$ is:

$$D_{KL}\bigl(q_\psi(z|x),\, p_\theta(z)\bigr) = -\frac{1}{2}\sum_j\left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

We scale the KL by $\frac{1}{N}$ (number of pixels $N = 784$) so it is in the same units as the per-pixel MSE.

In [ ]:
import torch.nn.functional as F

def elbo_loss(
    x: torch.Tensor,
    x_hat: torch.Tensor,
    mu: torch.Tensor,
    log_var: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Compute the negative ELBO as a sum of reconstruction loss and KL divergence.

    The KL term is scaled by the number of input pixels so both terms are
    in the same units as the per-pixel MSE.

    Returns:
        total  — sum of recon_loss and kl_loss (scalar to backprop)
        recon  — MSE reconstruction loss (for logging)
        kl     — KL divergence term (for logging)
    """
    n_pixels = x.shape[1] * x.shape[2] * x.shape[3]  # 1*28*28 = 784

    recon = F.mse_loss(x_hat, x, reduction="mean")
    kl    = (-0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).sum(dim=1)).mean() / n_pixels

    return recon + kl, recon, kl

### Training

We log both the reconstruction and KL term separately each epoch. At the start of training the KL term is close to zero — the encoder outputs an approximately standard normal posterior because the weights are initialised near zero. As training progresses, the reconstruction term decreases while the KL term rises slightly, reflecting the trade-off the model learns to balance.

In [ ]:
import torch.optim as optim

model     = VAE(latent_dim=LATENT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs    = 20

for epoch in range(epochs):
    model.train()
    total_loss = total_recon = total_kl = 0

    for x, _ in train_loader:
        x = x.to(device, non_blocking=True)

        optimizer.zero_grad()
        x_hat, mu, log_var = model(x)
        loss, recon, kl = elbo_loss(x, x_hat, mu, log_var)
        loss.backward()
        optimizer.step()

        total_loss  += loss.item()
        total_recon += recon.item()
        total_kl    += kl.item()

    n = len(train_loader)
    print(f"Epoch {epoch+1:2d}  "
          f"loss={total_loss/n:.5f}  "
          f"recon={total_recon/n:.5f}  "
          f"kl={total_kl/n:.5f}")

In [ ]:
model.save_model()

If you do not want to train, you can load the pre-trained model (latent_dim=2, 20 epochs, full MNIST training set).

In [ ]:
import torch
from VAE import VAE

device     = "cuda" if torch.cuda.is_available() else "cpu"
LATENT_DIM = 2

model = VAE(latent_dim=LATENT_DIM).to(device)
model.load_model(path="AIBIP/06-Generative_Image_Models/models/vae_mnist.pth", device=device)

### Reconstruction Quality

We compare originals and reconstructions for the first 10 test images. At `latent_dim=2` the VAE bottleneck is very tight; slight blurring is expected and is a direct consequence of the KL regularisation pushing posteriors towards the prior.

In [ ]:
import matplotlib.pyplot as plt

model.eval()

x_batch, _ = next(iter(test_loader))
x_batch    = x_batch[:10].to(device)

with torch.no_grad():
    x_hat, _, _ = model(x_batch)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    axes[0, i].imshow(x_batch[i].squeeze().cpu(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(x_hat[i].squeeze().cpu(), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original",       fontsize=10)
axes[1, 0].set_ylabel("Reconstruction", fontsize=10)
plt.tight_layout()
plt.show()

### Latent Space Visualisation

We encode all 10,000 test images and plot $\boldsymbol{\mu}$ — the posterior mean — in 2D, coloured by digit class. Compared to the plain autoencoder:

- Clusters are **smoother and more overlapping** because the KL term prevents the encoder from collapsing all representations to isolated points.
- The occupied region is roughly centred at the origin and has unit-scale spread, consistent with the $\mathcal{N}(\mathbf{0}, \mathbf{I})$ prior.

In [ ]:
import numpy as np

model.eval()

all_mu     = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        mu, _ = model.encoder(x.to(device))
        all_mu.append(mu.cpu().numpy())
        all_labels.append(y.numpy())

all_mu     = np.concatenate(all_mu,     axis=0)  # (10000, 2)
all_labels = np.concatenate(all_labels, axis=0)  # (10000,)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(all_mu[:, 0], all_mu[:, 1],
                     c=all_labels, cmap="tab10", s=2, alpha=0.6)
plt.colorbar(scatter, ax=ax, label="Digit class", ticks=range(10))
ax.set_xlabel("z[0]")
ax.set_ylabel("z[1]")
ax.set_title("2D Latent Space (posterior means) — MNIST Test Set")
plt.tight_layout()
plt.show()

### Latent Space Grid Decoding

Because the prior is $\mathcal{N}(\mathbf{0}, \mathbf{I})$, the latent space has a known scale. We can sweep a uniform grid over $[-3, 3]^2$ and decode each point, revealing how the VAE has organised digit morphology across the 2D plane.

In [ ]:
model.eval()

n_side  = 15
z_range = torch.linspace(-3, 3, n_side)

# Build grid of (n_side^2, 2) latent points
grid_z = torch.stack(
    torch.meshgrid(z_range, z_range, indexing="ij"), dim=-1
).reshape(-1, 2).to(device)

with torch.no_grad():
    imgs = model.decode(grid_z).squeeze(1).cpu().numpy()  # (n_side^2, 28, 28)

canvas = np.zeros((n_side * 28, n_side * 28))
for idx, img in enumerate(imgs):
    row = idx // n_side
    col = idx  % n_side
    canvas[row*28:(row+1)*28, col*28:(col+1)*28] = img

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(canvas, cmap="gray", extent=[-3, 3, -3, 3], origin="upper")
ax.set_xlabel("z[0]")
ax.set_ylabel("z[1]")
ax.set_title("Decoded images on a 15×15 grid in latent space")
plt.tight_layout()
plt.show()

### Sampling from the Prior

This is the key capability the plain autoencoder *lacks*: we can draw $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and decode to obtain new, previously unseen images. The KL regularisation ensures that points drawn from the prior land in regions the decoder has been trained on.

In [ ]:
model.eval()

n_samples = 16
samples   = model.sample(n_samples, device=device).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle("Samples from p(z) = N(0, I) decoded by the VAE", y=1.02)
plt.tight_layout()
plt.show()

### Latent Space Interpolation

As with the autoencoder, we can linearly interpolate between the posterior means of two test images. The VAE interpolation is typically smoother because the KL term encourages a continuous, gap-free latent space.

In [ ]:
model.eval()

test_images, test_labels = next(iter(test_loader))

idx_a = (test_labels == 1).nonzero(as_tuple=True)[0][0]
idx_b = (test_labels == 7).nonzero(as_tuple=True)[0][0]

x_a = test_images[idx_a].unsqueeze(0).to(device)
x_b = test_images[idx_b].unsqueeze(0).to(device)

with torch.no_grad():
    mu_a, _ = model.encoder(x_a)
    mu_b, _ = model.encoder(x_b)

n_steps = 10
alphas  = torch.linspace(0, 1, n_steps)

fig, axes = plt.subplots(1, n_steps, figsize=(15, 2))

with torch.no_grad():
    for i, alpha in enumerate(alphas):
        z_interp = (1 - alpha) * mu_a + alpha * mu_b
        img = model.decode(z_interp).squeeze().cpu()
        axes[i].imshow(img, cmap="gray")
        axes[i].axis("off")
        axes[i].set_title(f"{alpha:.1f}", fontsize=8)

plt.suptitle(f"Interpolation: digit {test_labels[idx_a].item()} → digit {test_labels[idx_b].item()}", y=1.05)
plt.tight_layout()
plt.show()

---
## Experiments with Fashion-MNIST

Fashion-MNIST is a drop-in replacement for MNIST: 60,000 training images of size 28×28, ten classes (T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot). The code is identical — only the dataset changes.

Fashion-MNIST is more challenging than MNIST: higher intra-class variance and less sharp inter-class boundaries. This makes the latent space structure more interesting to inspect and the generative quality of the VAE easier to judge visually.

In [ ]:
from torchvision.datasets import FashionMNIST

FASHION_CLASSES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",  "Shirt",   "Sneaker",  "Bag",   "Ankle boot"
]

fashion_train = FashionMNIST(root="./data", train=True,  download=True, transform=transform)
fashion_test  = FashionMNIST(root="./data", train=False, download=True, transform=transform)

fashion_train_loader = DataLoader(fashion_train, batch_size=256, shuffle=True,  num_workers=0, pin_memory=True)
fashion_test_loader  = DataLoader(fashion_test,  batch_size=256, shuffle=False, num_workers=0)

print(f"Training samples: {len(fashion_train)}, Test samples: {len(fashion_test)}")

### Training

Same architecture and hyperparameters as before. Because Fashion-MNIST is more complex, the reconstruction loss settles at a higher value — the 2D bottleneck is a tight constraint for ten visually diverse classes.

In [ ]:
fashion_model     = VAE(latent_dim=LATENT_DIM).to(device)
fashion_optimizer = optim.Adam(fashion_model.parameters(), lr=1e-3)

for epoch in range(epochs):
    fashion_model.train()
    total_loss = total_recon = total_kl = 0

    for x, _ in fashion_train_loader:
        x = x.to(device, non_blocking=True)

        fashion_optimizer.zero_grad()
        x_hat, mu, log_var = fashion_model(x)
        loss, recon, kl = elbo_loss(x, x_hat, mu, log_var)
        loss.backward()
        fashion_optimizer.step()

        total_loss  += loss.item()
        total_recon += recon.item()
        total_kl    += kl.item()

    n = len(fashion_train_loader)
    print(f"Epoch {epoch+1:2d}  "
          f"loss={total_loss/n:.5f}  "
          f"recon={total_recon/n:.5f}  "
          f"kl={total_kl/n:.5f}")

In [ ]:
fashion_model.save_model(path="models/vae_fashion_mnist.pth")

If you do not want to train, load the pre-trained model (latent_dim=2, 20 epochs, full Fashion-MNIST training set).

In [ ]:
fashion_model = VAE(latent_dim=LATENT_DIM).to(device)
fashion_model.load_model(path="AIBIP/06-Generative_Image_Models/models/vae_fashion_mnist.pth", device=device)

### Reconstruction Quality

Reconstructions are noticeably blurrier than on MNIST — a known limitation of VAEs with MSE loss on complex textures. The model captures the overall shape and class identity but loses fine detail.

In [ ]:
fashion_model.eval()

x_batch, labels_batch = next(iter(fashion_test_loader))
x_batch = x_batch[:10].to(device)

with torch.no_grad():
    x_hat, _, _ = fashion_model(x_batch)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    axes[0, i].imshow(x_batch[i].squeeze().cpu(), cmap="gray")
    axes[0, i].axis("off")
    axes[0, i].set_title(FASHION_CLASSES[labels_batch[i].item()], fontsize=6)
    axes[1, i].imshow(x_hat[i].squeeze().cpu(), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original",       fontsize=10)
axes[1, 0].set_ylabel("Reconstruction", fontsize=10)
plt.tight_layout()
plt.show()

### Latent Space

The 2D latent space shows less clean separation than MNIST — visually similar classes (Shirt vs. T-shirt, Sneaker vs. Ankle boot) overlap strongly. The KL regularisation still enforces a roughly standard-normal distribution.

In [ ]:
fashion_model.eval()

all_mu     = []
all_labels = []

with torch.no_grad():
    for x, y in fashion_test_loader:
        mu, _ = fashion_model.encoder(x.to(device))
        all_mu.append(mu.cpu().numpy())
        all_labels.append(y.numpy())

all_mu     = np.concatenate(all_mu,     axis=0)
all_labels = np.concatenate(all_labels, axis=0)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(all_mu[:, 0], all_mu[:, 1],
                     c=all_labels, cmap="tab10", s=2, alpha=0.6)
cbar = plt.colorbar(scatter, ax=ax, ticks=range(10))
cbar.ax.set_yticklabels(FASHION_CLASSES, fontsize=7)
ax.set_xlabel("z[0]")
ax.set_ylabel("z[1]")
ax.set_title("2D Latent Space (posterior means) — Fashion-MNIST Test Set")
plt.tight_layout()
plt.show()

### Sampling from the Prior

Samples drawn from $p_\theta(\mathbf{z}) = \mathcal{N}(\mathbf{0}, \mathbf{I})$ and decoded by the Fashion-MNIST VAE. The generated items are recognisable as clothing, though blurrier than real examples — typical for VAEs with pixel-wise MSE loss.

In [ ]:
fashion_model.eval()

samples = fashion_model.sample(16, device=device).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle("Samples from $p_\\theta(z) = \\mathcal{N}(0, I)$ — Fashion-MNIST VAE", y=1.02)
plt.tight_layout()
plt.show()